# Edit Tagger v2 Training (Colab)

이 노트북은 `aihub_highfreq_jsonl.tgz`, `build_edit_tagger_v2_dataset.py`, `train_edit_tagger_v2.py`를 Google Drive에 올려둔 상태를 기준으로, Colab free tier에서 Edit Tagger v2를 학습합니다.

핵심 변화:
- 라벨을 `KEEP / SPACE_FIX / OPEN_REPLACE`로 단순화
- clean KEEP 예시를 같이 넣어 false positive를 줄임
- 목표 지표를 `open_replace_recall` 중심으로 봄


## 0. 준비

- `런타임 > 런타임 유형 변경 > GPU`로 바꾸세요.
- Drive 경로는 `MyDrive/grammarly_korean/` 기준입니다.
- Drive에 아래 3개 파일이 있어야 합니다.
  - `aihub_highfreq_jsonl.tgz`
  - `build_edit_tagger_v2_dataset.py`
  - `train_edit_tagger_v2.py`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!ls -lah /content/drive/MyDrive/grammarly_korean


In [ ]:
!mkdir -p /content/work
!cp /content/drive/MyDrive/grammarly_korean/aihub_highfreq_jsonl.tgz /content/work/
!cp /content/drive/MyDrive/grammarly_korean/build_edit_tagger_v2_dataset.py /content/work/
!cp /content/drive/MyDrive/grammarly_korean/train_edit_tagger_v2.py /content/work/
!ls -lah /content/work


In [ ]:
!tar -xzf /content/work/aihub_highfreq_jsonl.tgz -C /content/work
!ls -lah /content/work/aihub_highfreq
!wc -l /content/work/aihub_highfreq/train.jsonl /content/work/aihub_highfreq/validation.jsonl


In [ ]:
!pip install -q transformers datasets accelerate seqeval optimum onnx onnxruntime


In [ ]:
!nvidia-smi


## 1. v2 데이터셋 생성

기본 설정은 noisy example과 clean KEEP example을 1:1로 섞습니다.


In [ ]:
!python /content/work/build_edit_tagger_v2_dataset.py \
  --input-dir /content/work/aihub_highfreq \
  --output-dir /content/work/edit_tagger_v2_dataset \
  --clean-ratio 1.0


In [ ]:
!ls -lah /content/work/edit_tagger_v2_dataset
!cat /content/work/edit_tagger_v2_dataset/summary.json


## 2. 1차 학습

free tier 기준으로 먼저 subset 학습을 권장합니다.
- train: 80,000
- validation: 8,000
- epoch: 2

체크포인트는 Drive에 직접 저장합니다.


In [ ]:
!python /content/work/train_edit_tagger_v2.py \
  --dataset-dir /content/work/edit_tagger_v2_dataset \
  --output-dir /content/drive/MyDrive/grammarly_korean/checkpoints/edit_tagger_v2_koelectra \
  --train-limit 80000 \
  --validation-limit 8000 \
  --batch-size 8 \
  --grad-accum 4 \
  --epochs 2 \
  --save-steps 500 \
  --eval-steps 500 \
  --logging-steps 100


In [ ]:
!ls -lah /content/drive/MyDrive/grammarly_korean/checkpoints/edit_tagger_v2_koelectra
!cat /content/drive/MyDrive/grammarly_korean/checkpoints/edit_tagger_v2_koelectra/metrics.json


## 3. 다음 판단 기준

먼저 아래 값을 봅니다.
- `open_replace_recall`
- `open_replace_precision`
- `keep_false_positive_rate`
- `space_fix_recall`

좋게 나오면 그 다음 단계는 두 가지입니다.
1. full train
2. ONNX export 후 서버 연결
